# 🏙️ Airbnb NYC 2019 — Complete Univariate & Bivariate Analysis
---
## 📋 Executive Summary

This notebook delivers a **production-grade EDA** of the Airbnb New York City 2019 dataset  
(~49 k listings, 16 features). Key findings at a glance:

| # | Insight |
|---|---------|
| 1 | **Price is heavily right-skewed** (skew ≈ 19) — median \$106 vs mean \$152; extreme luxury outliers distort averages |
| 2 | **Manhattan dominates** with 44 % of listings and the highest median price (\$150); Bronx is cheapest (\$65 median) |
| 3 | **Entire home/apt** commands a 2.3× price premium over private rooms; shared rooms are a niche (<2 %) |
| 4 | **`minimum_nights` has extreme outliers** (max 1 250 nights) — log-transform required before ML |
| 5 | **10 k listings (20 %) have never been reviewed** — a significant missingness pattern tied to low availability |

**Preprocessing Recommendations for ML:**  
`price` → log-transform | `minimum_nights` → log-transform | drop `id`, `name`, `host_name`, `last_review` |  
encode `neighbourhood_group`, `room_type` | impute `reviews_per_month` with 0 (no reviews = 0 rate)

---
**Dataset:** AB_NYC_2019.csv | **Rows:** 48,895 | **Columns:** 16  
**Libraries:** pandas · numpy · matplotlib · seaborn · scipy


## 1. 📥 Data Loading & Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings, os

warnings.filterwarnings('ignore')

# ── Aesthetics ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PALETTE   = 'muted'
CMAP      = 'YlOrRd'
FIG_COLOR = '#F8F9FA'
ACCENT    = '#E63946'

os.makedirs('plots', exist_ok=True)
print("✅ Libraries loaded")


In [ ]:
# ── Load data ───────────────────────────────────────────────────────────────
df = pd.read_csv('AB_NYC_2019.csv')

print(f"Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()


In [ ]:
# ── Data types overview ─────────────────────────────────────────────────────
df.info()


In [ ]:
# ── Missing values heatmap ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=FIG_COLOR)

# Bar chart of missing counts
miss = df.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=False)
axes[0].barh(miss.index, miss.values, color=ACCENT)
axes[0].set_xlabel('Missing Count')
axes[0].set_title('Missing Values per Column', fontweight='bold')
for i, v in enumerate(miss.values):
    axes[0].text(v + 10, i, f'{v:,} ({v/len(df)*100:.1f}%)', va='center', fontsize=9)

# Heatmap of nulls across sample
sample = df.isnull().sample(min(500, len(df)), random_state=42)
sns.heatmap(sample.T, cbar=False, cmap='Reds', ax=axes[1], yticklabels=True)
axes[1].set_title('Null Pattern (500-row sample)', fontweight='bold')
axes[1].set_xlabel('Row index (sample)')

plt.suptitle('Data Quality: Missing Values', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/00_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()


### 🔍 Data Quality Insights
- **`last_review` & `reviews_per_month`** share identical missingness (10,052 rows, ~20.6%) — these listings have **zero reviews**, so `reviews_per_month` should be imputed with **0**, not dropped.
- **`name` (16) & `host_name` (21)** have negligible nulls; these columns will be dropped for ML anyway.
- **No nulls** in key numeric targets: `price`, `availability_365`, `minimum_nights`. Clean for modelling.


## 2. 🔍 Univariate Analysis — Numerical Variables

In [ ]:
NUM_COLS = ['price', 'minimum_nights', 'number_of_reviews',
            'reviews_per_month', 'calculated_host_listings_count', 'availability_365']

# Summary statistics
stats_df = df[NUM_COLS].agg(['mean','median','std',
    lambda x: x.mode()[0],
    'min','max',
    lambda x: x.quantile(0.25),
    lambda x: x.quantile(0.75),
    lambda x: stats.skew(x.dropna()),
    lambda x: stats.kurtosis(x.dropna())
]).T
stats_df.columns = ['Mean','Median','Std','Mode','Min','Max','Q1','Q3','Skewness','Kurtosis']
stats_df = stats_df.round(2)
print("📊 Numerical Summary Statistics")
display(stats_df)


In [ ]:
# ── Histograms + KDE for all numerical variables ────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 13), facecolor=FIG_COLOR)
axes = axes.flatten()

for i, col in enumerate(NUM_COLS):
    data = df[col].dropna()
    skew_val = stats.skew(data)
    axes[i].hist(data.clip(upper=data.quantile(0.99)), bins=50,
                 color=sns.color_palette(PALETTE)[i], edgecolor='white', alpha=0.85)
    ax2 = axes[i].twinx()
    data.clip(upper=data.quantile(0.99)).plot.kde(ax=ax2, color=ACCENT, linewidth=2)
    ax2.set_ylabel('Density', fontsize=8)
    ax2.set_yticks([])
    axes[i].set_title(f'{col}  |  skew={skew_val:.2f}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

plt.suptitle('Univariate Distributions — Numerical Features (clipped at 99th pct)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/01_histograms_numerical.png', dpi=150, bbox_inches='tight')
plt.show()


### 🔍 Distribution Insights
- **`price`** skew ≈ 19: massively right-skewed. 75 % of listings price below \$175; a handful reach \$10,000. **→ Log-transform before ML.**
- **`minimum_nights`** skew ≈ 17: most listings require 1–3 nights; a long-stay segment (30–365 nights) pulls the tail. **→ Log-transform or cap outliers.**
- **`number_of_reviews`** is zero-inflated with heavy right tail — many new listings, a few with 600+ reviews.
- **`reviews_per_month`** bimodal hint: a spike at 0 (no reviews) and a peak around 1–2/month for active listings.
- **`availability_365`** bimodal: strong peaks at 0 (blocked/inactive) and 365 (always available) — two distinct host strategies.


In [ ]:
# ── Box plots for outlier visualisation ─────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9), facecolor=FIG_COLOR)
axes = axes.flatten()

for i, col in enumerate(NUM_COLS):
    data = df[col].dropna()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    n_outliers = ((data < q1 - 1.5*iqr) | (data > q3 + 1.5*iqr)).sum()
    axes[i].boxplot(data, vert=True, patch_artist=True,
                    boxprops=dict(facecolor=sns.color_palette(PALETTE)[i], alpha=0.7),
                    medianprops=dict(color=ACCENT, linewidth=2),
                    flierprops=dict(marker='.', markersize=2, alpha=0.3))
    axes[i].set_title(f'{col}\n{n_outliers:,} outliers ({n_outliers/len(data)*100:.1f}%)', 
                      fontweight='bold')
    axes[i].set_ylabel(col)

plt.suptitle('Box Plots — Outlier Identification', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/02_boxplots_numerical.png', dpi=150, bbox_inches='tight')
plt.show()


### 🔍 Outlier Insights
- **`price`**: ~3 % IQR-outliers (>\$355) but these represent real luxury listings, not data errors — winsorise or log-transform rather than drop.
- **`minimum_nights`**: 1,250-night maximum is likely policy anomalies (long-term rentals); cap at 365 for short-stay analysis.
- **`calculated_host_listings_count`**: a handful of professional hosts list 300+ units — a "superhost" skew worth a separate segment.


## 3. 🔍 Univariate Analysis — Categorical Variables

In [ ]:
CAT_COLS = ['neighbourhood_group', 'room_type']

for col in CAT_COLS:
    vc = df[col].value_counts()
    pct = (vc / len(df) * 100).round(1)
    summary = pd.DataFrame({'Count': vc, 'Percentage': pct})
    print(f"\n📌 {col}")
    display(summary)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=FIG_COLOR)

# neighbourhood_group
ng_counts = df['neighbourhood_group'].value_counts()
colors = sns.color_palette('muted', len(ng_counts))
bars = axes[0].barh(ng_counts.index, ng_counts.values, color=colors, edgecolor='white')
axes[0].set_title('Listings by Borough', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Number of Listings')
for bar, val in zip(bars, ng_counts.values):
    axes[0].text(val + 100, bar.get_y() + bar.get_height()/2,
                 f'{val:,} ({val/len(df)*100:.1f}%)', va='center', fontsize=9)

# room_type
rt_counts = df['room_type'].value_counts()
wedges, texts, autotexts = axes[1].pie(rt_counts.values, labels=rt_counts.index,
    autopct='%1.1f%%', colors=sns.color_palette('muted', len(rt_counts)),
    startangle=90, wedgeprops=dict(edgecolor='white', linewidth=1.5))
for at in autotexts: at.set_fontsize(10)
axes[1].set_title('Room Type Distribution', fontweight='bold', fontsize=13)

plt.suptitle('Categorical Variable Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/03_categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


### 🔍 Categorical Insights
- **Manhattan** holds 44.3 % of all listings — nearly half the market — making it the dominant borough for analysis.
- **Entire home/apt (52 %)** is the most common offering, signalling that many hosts rent their primary home rather than a single room.
- **Shared rooms (<2 %)** are a niche product; collapsing them with private rooms may be appropriate for ML to avoid class imbalance.


In [ ]:
# ── Top-20 neighbourhoods ────────────────────────────────────────────────────
top20 = df['neighbourhood'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(13, 7), facecolor=FIG_COLOR)
palette = sns.color_palette('YlOrRd_r', 20)
bars = ax.barh(top20.index[::-1], top20.values[::-1], color=palette[::-1], edgecolor='white')
ax.set_xlabel('Number of Listings')
ax.set_title('Top 20 Neighbourhoods by Listing Count', fontweight='bold', fontsize=13)
for bar, val in zip(bars, top20.values[::-1]):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('plots/04_top20_neighbourhoods.png', dpi=150, bbox_inches='tight')
plt.show()


### 🔍 Neighbourhood Insights
- **Williamsburg & Bedford-Stuyvesant** (Brooklyn) lead all neighbourhoods — affordable Brooklyn has become Airbnb's second city within NYC.
- **Harlem** ranks 3rd, reflecting the price-sensitive North Manhattan segment popular with budget travellers.
- These top-3 neighbourhoods alone account for ~8 % of all listings.


## 4. 🔗 Bivariate Analysis — Numerical vs Numerical

In [ ]:
# ── Correlation matrix ───────────────────────────────────────────────────────
corr = df[NUM_COLS].corr()

fig, ax = plt.subplots(figsize=(10, 8), facecolor=FIG_COLOR)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            annot_kws={'size': 11}, ax=ax)
ax.set_title('Correlation Matrix — Numerical Features', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('plots/05_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


### 🔍 Correlation Insights
- **`number_of_reviews` ↔ `reviews_per_month`** (r ≈ 0.55): the strongest correlation — older active listings naturally accumulate both.
- **`price` correlates weakly with everything** (max |r| ≈ 0.05) — price is largely driven by location & room type, not captured by numeric correlations alone.
- **`availability_365` ↔ `reviews_per_month`** (r ≈ 0.20): more available listings attract more reviews — a virtuous cycle for active hosts.
- **No severe multicollinearity** — all pairs well below |0.8|; no need to drop features for regression on numeric grounds alone.


In [ ]:
# ── Scatter: price vs number_of_reviews (log scale) ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=FIG_COLOR)

# Raw (capped for visibility)
axes[0].scatter(df['number_of_reviews'], df['price'].clip(upper=500),
                alpha=0.1, s=5, color=sns.color_palette('muted')[0])
m, b = np.polyfit(df['number_of_reviews'], df['price'].clip(upper=500), 1)
x_line = np.linspace(0, df['number_of_reviews'].max(), 200)
axes[0].plot(x_line, m*x_line + b, color=ACCENT, linewidth=2, label=f'Trend (slope={m:.3f})')
axes[0].set_xlabel('Number of Reviews')
axes[0].set_ylabel('Price (capped $500)')
axes[0].set_title('Price vs Reviews (Raw)', fontweight='bold')
axes[0].legend()

# Log-transformed price
log_price = np.log1p(df['price'])
axes[1].scatter(df['number_of_reviews'], log_price, alpha=0.1, s=5,
                color=sns.color_palette('muted')[1])
m2, b2 = np.polyfit(df['number_of_reviews'], log_price, 1)
axes[1].plot(x_line, m2*x_line + b2, color=ACCENT, linewidth=2, label=f'Trend (slope={m2:.4f})')
axes[1].set_xlabel('Number of Reviews')
axes[1].set_ylabel('log(Price + 1)')
axes[1].set_title('log(Price) vs Reviews', fontweight='bold')
axes[1].legend()

plt.suptitle('Price vs Number of Reviews', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/06_scatter_price_reviews.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Pearson r (price vs reviews): {df['price'].corr(df['number_of_reviews']):.3f}")


In [ ]:
# ── Pairplot — key features ──────────────────────────────────────────────────
sample_df = df[['price', 'minimum_nights', 'number_of_reviews',
                'availability_365', 'neighbourhood_group']].sample(3000, random_state=42)
# Log-transform for readability
sample_df['log_price'] = np.log1p(sample_df['price'])
sample_df['log_min_nights'] = np.log1p(sample_df['minimum_nights'])
plot_cols = ['log_price', 'log_min_nights', 'number_of_reviews', 'availability_365']

g = sns.pairplot(sample_df, vars=plot_cols, hue='neighbourhood_group',
                 palette='muted', plot_kws={'alpha': 0.3, 's': 15},
                 diag_kind='kde', diag_kws={'alpha': 0.6})
g.fig.suptitle('Pairplot — Key Features Coloured by Borough (3k sample)', 
               y=1.02, fontsize=13, fontweight='bold')
g.fig.set_size_inches(12, 10)
plt.savefig('plots/07_pairplot.png', dpi=120, bbox_inches='tight')
plt.show()


### 🔍 Pairplot Insights
- **Manhattan listings (blue)** cluster at higher `log_price` across all subplots — borough is a dominant pricing signal.
- **`log_price` vs `availability_365`**: slight upward drift — expensive listings may be kept open year-round as investment properties.
- **`log_min_nights` vs `log_price`**: listings with very high minimum nights (long-term rentals) span all price points, suggesting they are a separate product category.


## 5. 🔗 Bivariate Analysis — Categorical vs Numerical

In [ ]:
# ── Median price by neighbourhood_group ─────────────────────────────────────
grp_stats = df.groupby('neighbourhood_group')['price'].agg(
    Median='median', Mean='mean', Count='size', Std='std').sort_values('Median', ascending=False)
print("📊 Price by Borough:")
display(grp_stats.round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=FIG_COLOR)

# Box plot
order_ng = df.groupby('neighbourhood_group')['price'].median().sort_values(ascending=False).index
sns.boxplot(data=df[df['price'] < 500], x='neighbourhood_group', y='price',
            order=order_ng, palette='muted', ax=axes[0])
axes[0].set_title('Price Distribution by Borough\n(capped $500)', fontweight='bold')
axes[0].set_xlabel('Borough')
axes[0].set_ylabel('Nightly Price ($)')
axes[0].tick_params(axis='x', rotation=20)

# Violin plot
sns.violinplot(data=df[df['price'] < 500], x='neighbourhood_group', y='price',
               order=order_ng, palette='muted', inner='quartile', ax=axes[1])
axes[1].set_title('Price Density by Borough\n(Violin, capped $500)', fontweight='bold')
axes[1].set_xlabel('Borough')
axes[1].set_ylabel('Nightly Price ($)')
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('Price by Borough', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/08_price_by_borough.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Price by room type ───────────────────────────────────────────────────────
rt_stats = df.groupby('room_type')['price'].agg(
    Median='median', Mean='mean', Count='size').sort_values('Median', ascending=False)
print("📊 Price by Room Type:")
display(rt_stats.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=FIG_COLOR)

order_rt = rt_stats.index
sns.boxplot(data=df[df['price'] < 600], x='room_type', y='price',
            order=order_rt, palette='Set2', ax=axes[0])
axes[0].set_title('Price Distribution by Room Type\n(capped $600)', fontweight='bold')
axes[0].set_xlabel('Room Type'); axes[0].set_ylabel('Price ($)')

sns.violinplot(data=df[df['price'] < 600], x='room_type', y='price',
               order=order_rt, palette='Set2', inner='quartile', ax=axes[1])
axes[1].set_title('Price Density by Room Type\n(Violin)', fontweight='bold')
axes[1].set_xlabel('Room Type'); axes[1].set_ylabel('Price ($)')

plt.suptitle('Price by Room Type', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/09_price_by_room_type.png', dpi=150, bbox_inches='tight')
plt.show()


### 🔍 Categorical × Price Insights
- **Manhattan** median \$150 is 2.3× the Bronx median (\$65) — borough alone explains a large chunk of price variance.
- **Entire home/apt** median (\$160) is 2.3× private room (\$70) — room type is arguably the single most powerful price predictor.
- **Violin plots** reveal bimodal distributions in Manhattan and entire-home listings: a budget cluster (\$50–150) and a luxury cluster (\$200+).


In [ ]:
# ── Heatmap: Median price by Borough × Room Type ────────────────────────────
pivot = df.pivot_table(values='price', index='neighbourhood_group',
                       columns='room_type', aggfunc='median')
pivot = pivot.sort_values('Entire home/apt', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5), facecolor=FIG_COLOR)
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5,
            ax=ax, annot_kws={'size': 12})
ax.set_title('Median Nightly Price (\$) — Borough × Room Type', fontweight='bold', fontsize=13)
ax.set_xlabel('Room Type'); ax.set_ylabel('Borough')
plt.tight_layout()
plt.savefig('plots/10_heatmap_price_borough_roomtype.png', dpi=150, bbox_inches='tight')
plt.show()


### 🔍 Interaction Insights
- **Manhattan Entire home/apt** commands the highest median (\$190) — the classic tourist accommodation.
- **Staten Island Entire home/apt** (\$175) is surprisingly comparable to Manhattan, likely reflecting larger family homes at competitive rates.
- **Shared rooms** show the smallest borough spread (\$45–\$80) — sharing price is relatively location-insensitive.


In [ ]:
# ── Availability & reviews by borough ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=FIG_COLOR)

# Reviews by borough
sns.boxplot(data=df[df['number_of_reviews'] < 200], x='neighbourhood_group',
            y='number_of_reviews', order=order_ng, palette='muted', ax=axes[0])
axes[0].set_title('Number of Reviews by Borough', fontweight='bold')
axes[0].set_xlabel('Borough'); axes[0].set_ylabel('Number of Reviews')
axes[0].tick_params(axis='x', rotation=20)

# Availability by borough
sns.boxplot(data=df, x='neighbourhood_group', y='availability_365',
            order=order_ng, palette='muted', ax=axes[1])
axes[1].set_title('Availability (365 days) by Borough', fontweight='bold')
axes[1].set_xlabel('Borough'); axes[1].set_ylabel('Days Available / Year')
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('Engagement Metrics by Borough', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/11_engagement_by_borough.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. 🔗 Bivariate Analysis — Categorical vs Categorical

In [ ]:
# ── Crosstab: Borough × Room Type ────────────────────────────────────────────
ct = pd.crosstab(df['neighbourhood_group'], df['room_type'], normalize='index').round(3) * 100
print("📊 Room Type Mix by Borough (%):")
display(ct.round(1))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=FIG_COLOR)

# Stacked bar
ct_raw = pd.crosstab(df['neighbourhood_group'], df['room_type'])
ct_raw_pct = ct_raw.div(ct_raw.sum(axis=1), axis=0) * 100
ct_raw_pct.loc[order_ng].plot(kind='bar', stacked=True, ax=axes[0],
    color=sns.color_palette('Set2', 3), edgecolor='white')
axes[0].set_title('Room Type Mix by Borough (%)', fontweight='bold')
axes[0].set_xlabel('Borough'); axes[0].set_ylabel('Percentage (%)')
axes[0].tick_params(axis='x', rotation=25)
axes[0].legend(title='Room Type', bbox_to_anchor=(1, 1))

# Count heatmap
ct_count = pd.crosstab(df['neighbourhood_group'], df['room_type'])
sns.heatmap(ct_count, annot=True, fmt=',', cmap='Blues', linewidths=0.5, ax=axes[1])
axes[1].set_title('Listing Counts: Borough × Room Type', fontweight='bold')
axes[1].set_xlabel('Room Type'); axes[1].set_ylabel('Borough')

plt.suptitle('Borough × Room Type Relationship', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/12_borough_roomtype_crosstab.png', dpi=150, bbox_inches='tight')
plt.show()


### 🔍 Categorical × Categorical Insights
- **Manhattan has the highest share of entire home/apt (~60 %)** — tourists prefer the full-apartment experience in the premium borough.
- **Bronx and Staten Island have the highest private room share (~45–48 %)** — budget-conscious travellers and local residents sharing rooms.
- **Queens is the most balanced borough** — nearly equal split between entire home and private room, catering to diverse traveller types.


In [ ]:
# ── Countplot: Top 10 neighbourhoods by room type ───────────────────────────
top10_hoods = df['neighbourhood'].value_counts().head(10).index
df_top10 = df[df['neighbourhood'].isin(top10_hoods)]

fig, ax = plt.subplots(figsize=(14, 6), facecolor=FIG_COLOR)
hood_order = df_top10.groupby('neighbourhood').size().sort_values(ascending=False).index
sns.countplot(data=df_top10, x='neighbourhood', hue='room_type',
              order=hood_order, palette='Set2', ax=ax)
ax.set_title('Top 10 Neighbourhoods: Listing Count by Room Type', fontweight='bold', fontsize=13)
ax.set_xlabel('Neighbourhood'); ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=35)
ax.legend(title='Room Type')
plt.tight_layout()
plt.savefig('plots/13_top10_neighbourhood_roomtype.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. 📊 Advanced Visualizations

In [ ]:
# ── Log-price distribution before & after transformation ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=FIG_COLOR)

df['price'].clip(upper=1000).hist(bins=60, ax=axes[0], color=sns.color_palette('muted')[0],
                                   edgecolor='white')
axes[0].set_title(f'Price (Raw) | skew={stats.skew(df["price"]):.1f}', fontweight='bold')
axes[0].set_xlabel('Price ($)'); axes[0].set_ylabel('Frequency')

np.log1p(df['price']).hist(bins=60, ax=axes[1], color=sns.color_palette('muted')[2],
                            edgecolor='white')
axes[1].set_title(f'log(Price+1) | skew={stats.skew(np.log1p(df["price"])):.2f}', fontweight='bold')
axes[1].set_xlabel('log(Price + 1)'); axes[1].set_ylabel('Frequency')

plt.suptitle('Price: Before vs After Log Transformation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/14_price_log_transform.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Raw price  skewness: {stats.skew(df['price']):.2f}")
print(f"Log price  skewness: {stats.skew(np.log1p(df['price'])):.2f}")


In [ ]:
# ── Geographic price heatmap (scatter proxy) ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=FIG_COLOR)

# All listings coloured by price
sc = axes[0].scatter(df['longitude'], df['latitude'],
                     c=np.log1p(df['price']), cmap='YlOrRd',
                     s=1.5, alpha=0.4)
plt.colorbar(sc, ax=axes[0], label='log(Price+1)')
axes[0].set_title('NYC Listings — Colour = log(Price)', fontweight='bold')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

# Coloured by borough
borough_palette = dict(zip(df['neighbourhood_group'].unique(),
                           sns.color_palette('muted', 5)))
for borough, grp in df.groupby('neighbourhood_group'):
    axes[1].scatter(grp['longitude'], grp['latitude'],
                    c=[borough_palette[borough]], s=1.5, alpha=0.35, label=borough)
axes[1].set_title('NYC Listings — Colour = Borough', fontweight='bold')
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')
axes[1].legend(markerscale=6, title='Borough', bbox_to_anchor=(1, 1))

plt.suptitle('Geographic Distribution of Airbnb Listings', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/15_geographic_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


### 🔍 Geographic Insights
- **Manhattan price concentration** is visible as a warm cluster in the left-centre — the island literally glows orange/red.
- **Brooklyn and Queens** form dense clusters east of Manhattan with moderate pricing (yellow-orange).
- **Staten Island and the Bronx** are sparse and cool-coloured, confirming their budget positioning.


In [ ]:
# ── Price: Top-20 most expensive neighbourhoods ──────────────────────────────
top_price_hoods = (df.groupby('neighbourhood')['price']
                     .agg(median='median', count='size')
                     .query('count >= 50')  # minimum sample
                     .sort_values('median', ascending=False)
                     .head(20))

fig, ax = plt.subplots(figsize=(12, 7), facecolor=FIG_COLOR)
colors = sns.color_palette('YlOrRd', 20)[::-1]
bars = ax.barh(top_price_hoods.index[::-1], top_price_hoods['median'][::-1],
               color=colors, edgecolor='white')
ax.set_xlabel('Median Nightly Price ($)')
ax.set_title('Top 20 Most Expensive Neighbourhoods (min 50 listings)', fontweight='bold', fontsize=13)
for bar, val in zip(bars, top_price_hoods['median'][::-1]):
    ax.text(val + 2, bar.get_y() + bar.get_height()/2, f'${val:.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('plots/16_top_expensive_neighbourhoods.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Host listing count distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=FIG_COLOR)

# Distribution
df['calculated_host_listings_count'].clip(upper=50).hist(
    bins=50, ax=axes[0], color=sns.color_palette('muted')[3], edgecolor='white')
axes[0].set_title('Host Listing Count Distribution (capped 50)', fontweight='bold')
axes[0].set_xlabel('# Listings by Host'); axes[0].set_ylabel('# Hosts')

# Segment breakdown
bins = [0,1,2,5,10,50,9999]
labels = ['1','2','3-5','6-10','11-50','50+']
df['host_segment'] = pd.cut(df['calculated_host_listings_count'], bins=bins, labels=labels)
seg_counts = df['host_segment'].value_counts().sort_index()
axes[1].bar(seg_counts.index, seg_counts.values, color=sns.color_palette('muted', 6), edgecolor='white')
axes[1].set_title('Host Listing Segments', fontweight='bold')
axes[1].set_xlabel('Host Size Segment'); axes[1].set_ylabel('Number of Listings')
for i, (idx, val) in enumerate(seg_counts.items()):
    axes[1].text(i, val + 100, f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=9)

plt.suptitle('Host Portfolio Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/17_host_segments.png', dpi=150, bbox_inches='tight')
plt.show()


### 🔍 Host Insights
- **~52 % of listings come from single-listing hosts** — the traditional "rent your home" Airbnb model.
- **~11 % from hosts with 50+ listings** — professional property managers disproportionately control supply in premium areas.
- This bimodal host structure creates a natural categorical feature: `is_professional_host` (listings > 5).


## 8. 🧠 Key Insights & ML Preprocessing Recommendations

---

### 🏆 Top 5 Business Insights

1. **Location × Room Type = Price**: The combination of borough and room type explains the majority of price variance.  
   Manhattan entire home median (\$190) is **4× the Bronx private room median (~\$48)**. Encode both as features.

2. **20 % "Ghost Listings"**: 10,052 listings (20.6 %) have zero reviews and missing `reviews_per_month`.  
   These are either new listings or inactive ones — a useful binary flag `is_new_listing` for ML.

3. **Superhost Effect**: ~11 % of listings belong to hosts with 50+ properties.  
   These professional operators price differently and have higher availability — segment them with `is_professional_host`.

4. **Bimodal Availability**: Availability clusters at 0 (blocked) and 365 (always open).  
   Bucketing into `low / medium / high` availability bands may outperform raw availability in tree models.

5. **Right-skew Everywhere**: `price`, `minimum_nights`, `number_of_reviews`, and `calculated_host_listings_count`  
   all exhibit severe right skew. Linear models will underperform without log-transformation.

---

### ⚙️ Preprocessing Recommendations for ML

| Column | Action | Rationale |
|--------|--------|-----------|
| `price` | `log1p()` transform | skew ≈ 19; normalise for regression |
| `minimum_nights` | `log1p()` + cap at 365 | skew ≈ 17; long-stay outliers |
| `number_of_reviews` | `log1p()` | zero-inflated, right-skewed |
| `reviews_per_month` | fill NaN → 0 | missingness = no reviews |
| `last_review` | drop or encode as `days_since_review` | date string; raw form not useful |
| `id`, `name`, `host_id`, `host_name` | **drop** | identifiers / free text |
| `neighbourhood_group`, `room_type` | **one-hot encode** | low cardinality categoricals |
| `neighbourhood` | **target encode** or top-N OHE | high cardinality (221 values) |
| `latitude`, `longitude` | keep or create cluster features | geographic signal |
| `calculated_host_listings_count` | `log1p()` + `is_professional_host` flag | right-skewed + segmentation value |
| `availability_365` | keep + optionally bucket | mildly bimodal |

---

### 🚀 Next Steps for ML

```
1. Feature Engineering
   - days_since_last_review = (reference_date - last_review)
   - is_new_listing = (number_of_reviews == 0).astype(int)
   - is_professional_host = (calculated_host_listings_count > 5).astype(int)
   - price_per_min_night = price / minimum_nights

2. Modelling Path
   - Baseline: Linear Regression on log(price)
   - Gradient Boosting (XGBoost / LightGBM) — handles skew & interactions natively
   - Evaluation: RMSE on log scale → exponentiate for interpretability

3. Clustering Extension
   - DBSCAN on (latitude, longitude) to create geo-cluster features
   - K-Means on host behaviour (reviews, availability, listings) for host typology
```


In [ ]:
# ── Final summary card ───────────────────────────────────────────────────────
print("=" * 60)
print("       AIRBNB NYC 2019 — EDA SUMMARY CARD")
print("=" * 60)
print(f"  Total listings        : {len(df):,}")
print(f"  Boroughs              : {df['neighbourhood_group'].nunique()}")
print(f"  Unique neighbourhoods : {df['neighbourhood'].nunique()}")
print(f"  Room types            : {df['room_type'].nunique()}")
print(f"  Unique hosts          : {df['host_id'].nunique():,}")
print()
print(f"  Median price          : ${df['price'].median():.0f}/night")
print(f"  Mean   price          : ${df['price'].mean():.0f}/night")
print(f"  Price skewness        : {stats.skew(df['price']):.1f}")
print()
print(f"  Listings w/o reviews  : {(df['number_of_reviews']==0).sum():,} ({(df['number_of_reviews']==0).mean()*100:.1f}%)")
print(f"  Professional hosts    : {(df['calculated_host_listings_count']>5).sum():,} ({(df['calculated_host_listings_count']>5).mean()*100:.1f}%)")
print()
print("  Plots saved to: ./plots/")
print("=" * 60)
